# Data Preparation Pipeline

This notebook builds the cleaned firm-year panel for ASEAN Green Bonds research and runs optional diagnostics.

**Flow**
1. Settings
2. Build full panel (pipeline)
3. Optional PSM integration (modular)
4. Survivorship diagnostics
5. ESG coverage
6. Summary



In [1]:
# === Settings ===
ALLOW_FX_FETCH = False  # honor: fx fetching disabled
SURVIVORSHIP_MODE = "exclude"  # ignore | exclude | weight
RUN_DIAGNOSTICS = True
PSM_MODE = "auto"  # auto | off | force
SCALING_METHOD = "robust"  # pipeline currently uses robust scaling



In [2]:
# === Imports ===
from pathlib import Path

import numpy as np
import pandas as pd

from asean_green_bonds import config
from asean_green_bonds.data import prepare_full_panel_data



In [3]:
# === Helper Functions ===

def detect_psm_inputs():
    auth_paths = [
        Path("data/green_bonds_with_authenticity_score.csv"),
        Path("data/green_bonds_authenticated.csv"),
    ]
    auth_path = next((p for p in auth_paths if p.exists()), None)
    raw_path = Path("data/green-bonds.csv")
    gb_raw_path = raw_path if raw_path.exists() else None
    return auth_path, gb_raw_path


def run_psm_pipeline(panel_df, psm_mode="auto"):
    # Integrate PSM attributes into the panel if inputs are available.
    if psm_mode == "off":
        print("PSM: disabled by PSM_MODE='off'.")
        return panel_df

    auth_path, gb_raw_path = detect_psm_inputs()
    psm_available = auth_path is not None
    if not psm_available:
        if psm_mode == "force":
            raise FileNotFoundError("Missing PSM input file. Provide authenticity file or set PSM_MODE='off'.")
        print("PSM: inputs not found, skipping.")
        return panel_df

    from asean_green_bonds.data.feature_engineering import (
        engineer_psm_attributes,
        merge_psm_into_panel,
        normalize_psm_attributes,
    )

    print()
    print("=" * 60)
    print("PSM ATTRIBUTES (OPTIONAL)")
    print("=" * 60)

    gb_auth = pd.read_csv(auth_path)
    gb_raw = pd.read_csv(gb_raw_path) if gb_raw_path is not None else gb_auth

    gb_engineered, _ = engineer_psm_attributes(gb_auth, gb_raw, verbose=True)
    gb_output_path = Path("data/green_bonds_with_psm_features.csv")
    gb_engineered.to_csv(gb_output_path, index=False)

    # Build market_df for org_permid mapping
    market_files = list(Path("data").glob("*-market.csv"))
    if Path("data/other-market.csv").exists():
        market_files.append(Path("data/other-market.csv"))
    market_dfs = [pd.read_csv(p) for p in market_files if p.exists()]
    market_df = (
        pd.concat(market_dfs, ignore_index=True)
        if market_dfs
        else pd.DataFrame(columns=["ric", "org_permid"])
    )

    panel_df = merge_psm_into_panel(panel_df, gb_engineered, market_df, verbose=True)
    panel_df = normalize_psm_attributes(panel_df, verbose=False)

    psm_cols = [
        "has_green_framework",
        "asset_tangibility",
        "issuer_track_record",
        "prior_green_bonds",
    ]
    present = [c for c in psm_cols if c in panel_df.columns]
    print(f"PSM columns in panel: {len(present)}/{len(psm_cols)}")

    return panel_df


def run_survivorship_diagnostics(panel_df, survivorship_mode="ignore"):
    from asean_green_bonds.data import prepare_analysis_sample, filter_survived_firms

    cfg = config.SURVIVORSHIP_CONFIG
    recent_years = cfg.get("recent_years", [2023, 2024, 2025])
    min_recent_observations = cfg.get("min_recent_observations", 1)
    existence_col = cfg.get("existence_col", "total_assets")
    early_years = cfg.get("early_years", [2015, 2016, 2017])

    print()
    print("=" * 60)
    print("SURVIVORSHIP BIAS ANALYSIS")
    print("=" * 60)
    print(f"Checking firm presence in recent years: {recent_years}")

    all_firms = panel_df["ric"].nunique() if "ric" in panel_df.columns else 0
    survived_df = filter_survived_firms(
        panel_df,
        firm_col="ric",
        recent_years=recent_years,
        min_recent_observations=min_recent_observations,
        existence_col=existence_col,
    )
    survived_firms = survived_df["ric"].nunique() if "ric" in survived_df.columns else 0
    dropped_firms = all_firms - survived_firms

    print(f"Total firms in panel: {all_firms}")
    print(f"Firms with recent data (survived): {survived_firms}")
    if all_firms > 0:
        print(f"Firms without recent data (potential attrition): {dropped_firms} ({100*dropped_firms/all_firms:.1f}%)")

    analysis_panel = prepare_analysis_sample(
        panel_df,
        survivorship_mode=survivorship_mode,
        firm_col="ric",
        time_col="Year",
        recent_years=recent_years,
        early_years=early_years,
        min_recent_observations=min_recent_observations,
        existence_col=existence_col,
    )

    print(f"Survivorship mode: {survivorship_mode}")
    print(f"Analysis panel size: {analysis_panel.shape}")

    if survivorship_mode == "weight" and "survivorship_weight" in analysis_panel.columns:
        print("Weight statistics:")
        print(f"  Mean: {analysis_panel['survivorship_weight'].mean():.3f}")
        print(f"  Range: [{analysis_panel['survivorship_weight'].min():.3f}, {analysis_panel['survivorship_weight'].max():.3f}]")

    return analysis_panel


def run_esg_coverage():
    from asean_green_bonds.authenticity import get_esg_coverage_by_country

    print()
    print("=" * 60)
    print("ESG DATA COVERAGE BY COUNTRY")
    print("=" * 60)

    esg_path = Path("data/esg_data.csv")
    gb_path = Path("data/green_bonds_authenticated.csv")
    if not esg_path.exists() or not gb_path.exists():
        print("ESG coverage: missing input files, skipping.")
        return

    esg_df = pd.read_csv(esg_path)
    gb_df = pd.read_csv(gb_path)

    esg_coverage = get_esg_coverage_by_country(esg_df, gb_df)
    print(esg_coverage.to_string(index=False))

    total_bonds = esg_coverage["total_bonds"].sum()
    tier1_bonds = esg_coverage["bonds_with_complete_esg"].sum()
    tier2_bonds = esg_coverage["bonds_with_partial_esg"].sum()
    tier3_bonds = esg_coverage["bonds_with_no_esg"].sum()

    if total_bonds > 0:
        print()
        print("Overall Tier Distribution:")
        print(f"  Tier 1 (Complete ESG): {tier1_bonds} ({100*tier1_bonds/total_bonds:.1f}%)")
        print(f"  Tier 2 (Partial ESG): {tier2_bonds} ({100*tier2_bonds/total_bonds:.1f}%)")
        print(f"  Tier 3 (No ESG data): {tier3_bonds} ({100*tier3_bonds/total_bonds:.1f}%)")


def print_panel_summary(df, label="Panel"):
    print()
    print("=" * 60)
    print(f"{label} SUMMARY")
    print("=" * 60)
    print(f"Shape: {df.shape}")
    if "ric" in df.columns:
        print(f"Entities (firms): {df['ric'].nunique()}")
    if "Year" in df.columns:
        print(f"Time periods: {df['Year'].nunique()}")
    if "green_bond_issue" in df.columns:
        print(f"Green bond issuers: {(df['green_bond_issue'] == 1).sum()}")
    if "has_green_bonds" in df.columns:
        print(f"Green bond issuer-years: {(df['has_green_bonds'] == 1).sum()}")

    missing = df.isnull().sum().sum()
    total = df.shape[0] * df.shape[1]
    print(f"Missing cells: {missing}")
    print(f"Missing rate: {missing / total * 100:.2f}%")



## Build Full Panel



In [4]:
print("Building cleaned full panel...")
panel_full = prepare_full_panel_data(
    survivorship_mode="ignore",
    allow_fx_fetch=ALLOW_FX_FETCH,
)
print(f"Cleaned full panel: {panel_full.shape}")

if SCALING_METHOD != "robust":
    print("Note: pipeline currently applies robust scaling. Update processing.py if you want a different method.")



Building cleaned full panel...
Cleaned full panel: (25462, 158)


## PSM (Optional)



In [5]:
panel_full = run_psm_pipeline(panel_full, psm_mode=PSM_MODE)




PSM ATTRIBUTES (OPTIONAL)
ENGINEERING MISSING PSM ATTRIBUTES

Step 1: Standardizing existing attributes (lowercase)...
  ✓ has_green_framework: 328 issuers
  ✓ issuer_track_record: min=0, max=65

Step 2: Engineering prior_green_bonds from issuance history...
  ✓ prior_green_bonds: 0 to 65
    - First-time issuers: 70
    - Repeat issuers: 263

Step 3: Estimating asset_tangibility by sector...
  ✓ asset_tangibility: 0.626 (mean)
    Range: 0.55 - 0.85
    Sectors mapped: 139
    Using fallback: 194

VERIFICATION: All PSM Attributes Present
  ✓ has_green_framework            - 333/333 non-null
  ✓ asset_tangibility              - 333/333 non-null
  ✓ issuer_track_record            - 333/333 non-null
  ✓ prior_green_bonds              - 333/333 non-null

✅ SUCCESS: All 4 PSM attributes engineered!
MERGING PSM ATTRIBUTES INTO FINAL PANEL

Step 1: Added org_permid: 25706/25718 non-null

Step 2: Building issuer-year PSM attributes (time-varying)...
  ✓ Issuer-year rows: 91
  ✓ Issuers with 

## Survivorship Diagnostics



In [6]:
panel_analysis = None
if RUN_DIAGNOSTICS:
    panel_analysis = run_survivorship_diagnostics(panel_full, survivorship_mode=SURVIVORSHIP_MODE)




SURVIVORSHIP BIAS ANALYSIS
Checking firm presence in recent years: [2023, 2024, 2025]
Total firms in panel: 4333
Firms with recent data (survived): 4318
Firms without recent data (potential attrition): 15 (0.3%)
Survivorship mode: exclude
Analysis panel size: (25673, 163)


## ESG Coverage (Optional)



In [7]:
if RUN_DIAGNOSTICS:
    run_esg_coverage()




ESG DATA COVERAGE BY COUNTRY


/Users/bunnypro/miniconda3/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


    country  total_bonds  bonds_with_complete_esg  bonds_with_partial_esg  bonds_with_no_esg  coverage_rate
  Australia            2                        0                       0                  2           0.00
  Hong Kong            1                        0                       0                  1           0.00
      India            1                        0                       0                  1           0.00
  Indonesia           25                        0                       0                 25           0.00
       Laos            3                        0                       0                  3           0.00
   Malaysia          125                        2                       0                123           1.60
Philippines           80                        0                       0                 80           0.00
  Singapore           34                        0                       0                 34           0.00
South Korea            1    

## Summary



In [8]:
print_panel_summary(panel_full, label="Full Panel")
if panel_analysis is not None and panel_analysis is not panel_full:
    print_panel_summary(panel_analysis, label=f"Analysis Panel ({SURVIVORSHIP_MODE})")

# Quick peek (avoid full table dump)
panel_full.head()




Full Panel SUMMARY
Shape: (25718, 163)
Entities (firms): 4333
Time periods: 6
Green bond issuers: 46
Green bond issuer-years: 46
Missing cells: 1893963
Missing rate: 45.18%

Analysis Panel (exclude) SUMMARY
Shape: (25673, 163)
Entities (firms): 4318
Time periods: 6
Green bond issuers: 46
Green bond issuer-years: 46
Missing cells: 1890503
Missing rate: 45.18%


,ric,Year,capital_expenditures,cash,current_assets_total,current_liabilities_total,earnings_bef_interest_tax,employees,interest_expense_total,long_term_debt,...,z_L1_esg_score,has_esg_score,esg_years_covered,green_bond_active,certified_bond_active,org_permid,has_green_framework,issuer_track_record,prior_green_bonds,asset_tangibility
0,1085.HK,2020,12624.307613,641777.452164,1.480179e+06,348768.795205,61107.884865,772.0,NaN,509.758085,...,NaN,0,0,0,0,4.295864e+09,0,0,0,0.55
1,1085.HK,2021,15327.882818,406254.023119,1.705442e+06,504300.965873,59741.005719,848.0,NaN,79.642765,...,NaN,0,0,0,0,4.295864e+09,0,0,0,0.55
2,1085.HK,2022,41522.728971,638541.337930,1.506691e+06,458767.149873,63711.897394,936.0,NaN,2107.796922,...,NaN,0,0,0,0,4.295864e+09,0,0,0,0.55
3,1085.HK,2023,17710.870828,772045.620363,1.874948e+06,483195.779579,110851.939852,1095.0,NaN,625773.428291,...,NaN,0,0,0,0,4.295864e+09,0,0,0,0.55
4,1085.HK,2024,12687.683578,947207.579723,2.191672e+06,920360.016154,100396.011527,902.0,NaN,804095.418784,...,NaN,0,0,0,0,4.295864e+09,0,0,0,0.55
